In [12]:
mol_type = "6r7m"

x = "Vector{Float64}([])"
bnds = 200.0
delaunay_eps = 100.0
overlap_jump = 0.0
overlap_slope = 1.1
rs = 1.4
η = 0.3665
temperature = 0.0

comment = "tasp_scan"
comment = replace(comment, " " => "_")

T_search_runs = 12
T_search_time = 15.0

simulation_time_minutes = 240.0
algorithm = "sa"
sa_level = "[1.0,0.5,0.2,0.5,0.3,0.0]"
energies = ["tasp"]
perturbation = "single_random"
initialization = "random"

n_mols = [2]
σ_rs = [0.25]
σ_ts = [1.25]
persistence_weights = ["[1.0,$(p1),$(p2)]" for p1 in -2.0:0.1:0.0 for p2 in 0.0:0.1:2.0];
#persistence_weights = ["[1.0,-0.25,-1.0]"]

In [13]:
simulations_per_combination = 1

input_specifier = "$(algorithm)_$(mol_type)_$(comment)"
output_directory = "../Simulations/unsorted_output/$(input_specifier)/"

open("../configs/$(input_specifier)_config.txt", "w") do io
    i = 1
    println(io,"ArrayTaskID input_string")
    output_directory = "../Simulations/unsorted_output/$(input_specifier)/"
    for _ in 0:simulations_per_combination-1, energy in energies, n_mol in n_mols, σ_r in σ_rs, σ_t in σ_ts, pws in persistence_weights
        name = "$(i)"
        input_string = escape_string("name=\"$name\";x=$(x);temperature=$(temperature);alg=\"$(algorithm)\";sa_level=$(sa_level);T_search_runs=$(T_search_runs);T_search_time=$(T_search_time);rs=$rs;η=$η;σ_t=$σ_t;σ_r=$σ_r;overlap_jump=$overlap_jump;overlap_slope=$overlap_slope;bnds=$bnds;persistence_weights=$pws;n_mol=$n_mol;mol_type=\"$mol_type\";nrg=\"$energy\";prtbt=\"$perturbation\";intl=\"$initialization\";output_directory=\"$output_directory\";delaunay_eps=$delaunay_eps;comment=\"$comment\";simulation_time_minutes=$simulation_time_minutes")
        println(io, "$i $input_string")
        i += 1
    end
end
total_simulations = length(readlines("../configs/$(input_specifier)_config.txt")) - 1

861

In [14]:
total_time_needed = simulation_time_minutes + (T_search_runs * T_search_time)

hours = Int(round(total_time_needed / 60.0))
days = hours ÷ 24
remaining_hours = hours % 24
remaining_hours_string = remaining_hours < 10 ? "0$(remaining_hours)" : string(remaining_hours)
buffer_time_string = total_time_needed < 5 ? "0$(Int(total_time_needed)+2)" : "30"

open("../$(input_specifier)_script.job", "w") do io
    println(io, "#!/bin/bash")
    println(io, "#SBATCH --job-name=SolvationSimulations")
    println(io, "#SBATCH --time=0$(days)-$(remaining_hours_string):$(buffer_time_string)")
    println(io, "#SBATCH --ntasks=1")
    println(io, "#SBATCH --cpus-per-task=1")
    println(io, "#SBATCH --array=1-$(total_simulations)")
    println(io, "#SBATCH --chdir=/work/spirandelli/MorphoMolHPC/")
    println(io, "#SBATCH -o ./job_log/$(input_specifier)/%a.out # STDOUT")
    println(io, "")
    println(io, "export http_proxy=http://proxy2.uni-potsdam.de:3128 #Setting proxy, due to lack of Internet on compute nodes.")
    println(io, "export https_proxy=http://proxy2.uni-potsdam.de:3128")
    println(io, "")
    println(io, "module purge")
    println(io, "source ../oineus_venv/bin/activate")    
    println(io, "module load devel/CMake/3.27.6-GCCcore-13.2.0")
    println(io, "module load devel/Boost/1.83.0-GCC-13.2.0")
    println(io, "module load lang/Julia/1.7.3-linux-x86_64")    
    println(io, "")
    println(io, "# Specify the path to the config file")
    println(io, "config=hpc_scripts/configs/$(input_specifier)_config.txt")
    println(io, "")
    println(io, "# Extract the variables from config file")
    println(io, "config_string=\$(awk -v ArrayTaskID=\$SLURM_ARRAY_TASK_ID '\$1==ArrayTaskID {print \$2}' \$config)")
    println(io, "")
    println(io, "julia -e \"$(escape_string("include(\"julia_scripts/generic_call.jl\"); generic_call(\"\$config_string\")"))\"")
end